# Customer Churn Prediction: EDA & Business Insights

This notebook documents the exploratory analysis for the IBM Telco Customer Churn dataset. The goal is not only to inspect the data, but to translate customer patterns into business decisions that can support retention strategy.

**Business question:** Can we identify customer groups with elevated churn risk early enough for the business to intervene?

## 1. Business Framing

Customer churn is a revenue protection problem. A useful churn project should explain who is leaving, why the patterns matter, and which retention actions should follow.

The EDA focuses on four business angles:

- Contract risk: Are flexible contracts associated with higher churn?
- Customer lifecycle: Are newer customers more likely to leave?
- Billing/payment friction: Are certain payment methods associated with churn?
- Service experience: Do support and service features affect churn risk?

## 2. Feature Hypotheses

Before modeling, it is useful to define expectations. This helps the analysis look intentional rather than like random charting.

| Feature | Expected effect on churn | Business reasoning |
|---|---|---|
| Contract | Month-to-month likely increases churn | Customers can leave more easily when they are not locked into longer contracts. |
| tenure | Short tenure likely increases churn | New customers may not yet be attached to the product or may still be evaluating alternatives. |
| MonthlyCharges | Higher charges may increase churn | Customers paying more may be more price-sensitive or expect better service quality. |
| PaymentMethod | Electronic check may indicate higher risk | Payment friction or manual payment behavior may correlate with dissatisfaction. |
| TechSupport | No tech support may increase churn | Customers without support may have unresolved service issues. |
| OnlineSecurity | No online security may increase churn | Lower service bundling can reduce perceived product value. |

## 3. Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 4. Load Dataset

In [ ]:
DATA_PATH = Path("../data/raw/telco_customer_churn.csv")
df = pd.read_csv(DATA_PATH)

## 5. Initial Data Checks

These checks show the dataset size, sample records, column types, missing values, and churn target distribution.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df["Churn"].value_counts()

In [ ]:
df["Churn"].value_counts(normalize=True).mul(100).round(2)

### Interpretation: Target Distribution

The churn target is imbalanced: most customers did not churn, while a smaller but commercially important group did. This means accuracy alone is not enough for model evaluation. A model can look accurate by mostly predicting "No churn," while still missing customers who are likely to leave.

For this business problem, recall, precision, F1 score, ROC-AUC, and the confusion matrix matter because the company needs to identify likely churners before revenue is lost.

## 6. Data Cleaning Rationale

`TotalCharges` appears as an object/text column in the raw dataset. For modeling and numerical analysis, it must be converted to a numeric field.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(20)

### Interpretation: Missing `TotalCharges`

The missing `TotalCharges` values occur for customers with `tenure = 0`. These customers have not accumulated billable tenure yet, so filling `TotalCharges` with `0` is a reasonable business assumption.

This is not a blind missing-value fill. The value is tied to the account lifecycle: no tenure means no accumulated total charges.

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df["ChurnFlag"] = df["Churn"].map({"No": 0, "Yes": 1})

df[["TotalCharges", "ChurnFlag"]].isnull().sum()

## 7. Target Leakage & Modeling Scope

Before modeling, we need to separate safe predictive features from columns that should not be used.

| Column | Use in model? | Reason |
|---|---|---|
| `customerID` | No | Identifier only. It does not describe customer behavior and can add noise. |
| `Churn` | No as feature | This is the target variable. Using it as an input would be target leakage. |
| `ChurnFlag` | No as feature | Encoded version of the target, used only for analysis/model target. |
| Service, contract, billing, tenure, charges fields | Yes | These are known customer/account attributes that can support prediction. |

The modeling pipeline drops `customerID` and uses `Churn` only as the supervised learning target.

In [ ]:
identifier_columns = ["customerID"]
target_columns = ["Churn", "ChurnFlag"]
safe_feature_columns = [col for col in df.columns if col not in identifier_columns + target_columns]

safe_feature_columns

## 8. Helper Function for Segment Churn

In [ ]:
def churn_rate_by(column):
    return (
        df.groupby(column)
        .agg(
            customers=("customerID", "count"),
            churned=("ChurnFlag", "sum"),
            churn_rate=("ChurnFlag", "mean"),
            avg_monthly_charge=("MonthlyCharges", "mean"),
        )
        .assign(
            churn_rate=lambda data: data["churn_rate"].mul(100).round(2),
            avg_monthly_charge=lambda data: data["avg_monthly_charge"].round(2),
        )
        .sort_values("churn_rate", ascending=False)
    )

## 9. Churn Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn", hue="Churn", palette=["#2563eb", "#dc2626"], legend=False)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customer Count")
plt.show()

### Interpretation: Churn Distribution

Most customers are retained, but the churned group is large enough to represent a meaningful revenue risk. The imbalance means the model should be judged by how well it identifies churners, not just by overall accuracy.

**Business action:** retention teams should use risk scoring to prioritize outreach instead of treating all customers equally.

## 10. Contract Analysis

In [ ]:
contract_churn = churn_rate_by("Contract")
contract_churn

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=contract_churn.reset_index(), x="Contract", y="churn_rate", color="#2563eb")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.show()

### Interpretation: Contract Type

Month-to-month customers have the highest churn rate. This supports the hypothesis that customers with fewer switching barriers are more likely to leave.

**Business action:** prioritize month-to-month customers for annual plan incentives, loyalty discounts, or bundled value offers before they churn.

## 11. Tenure Analysis

In [ ]:
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 months", "13-24 months", "25-48 months", "49-72 months"],
)

tenure_group_churn = churn_rate_by("TenureGroup")
tenure_group_churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="tenure", hue="Churn", fill=True, common_norm=False, alpha=0.35)
plt.title("Tenure Distribution by Churn Outcome")
plt.xlabel("Tenure in Months")
plt.show()

### Interpretation: Tenure

Churn is concentrated among customers with shorter tenure. This suggests churn prevention should start early in the customer lifecycle, not only when a customer is already close to cancelling.

**Business action:** create first-90-day and first-12-month onboarding journeys, satisfaction checks, and support touchpoints for new customers.

## 12. Payment Method Analysis

In [ ]:
payment_churn = churn_rate_by("PaymentMethod")
payment_churn

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=payment_churn.reset_index(), x="PaymentMethod", y="churn_rate", color="#059669")
plt.title("Churn Rate by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=30, ha="right")
plt.show()

### Interpretation: Payment Method

Electronic check customers show elevated churn compared with automatic payment methods. This may indicate billing friction, lower commitment, or a customer segment that is less embedded in recurring payment behavior.

**Business action:** encourage high-risk electronic check customers to switch to automatic payments using small credits, payment reminders, or simplified billing flows.

## 13. Internet Service Analysis

In [ ]:
internet_churn = churn_rate_by("InternetService")
internet_churn

In [ ]:
plt.figure(figsize=(7, 4))
sns.barplot(data=internet_churn.reset_index(), x="InternetService", y="churn_rate", color="#7c3aed")
plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Churn Rate (%)")
plt.show()

### Interpretation: Internet Service

Fiber optic customers show higher churn than DSL and customers without internet service. This does not automatically mean fiber is bad; it may reflect higher prices, higher customer expectations, or competitive alternatives.

**Business action:** investigate fiber optic customer complaints, pricing sensitivity, service reliability, and competitor offers before designing retention campaigns.

## 14. Support & Service Add-ons

In [ ]:
service_columns = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]
service_churn = pd.concat({col: churn_rate_by(col)["churn_rate"] for col in service_columns}, axis=1)
service_churn

In [ ]:
tech_support_churn = churn_rate_by("TechSupport")

plt.figure(figsize=(7, 4))
sns.barplot(data=tech_support_churn.reset_index(), x="TechSupport", y="churn_rate", color="#ea580c")
plt.title("Churn Rate by Tech Support Status")
plt.xlabel("Tech Support")
plt.ylabel("Churn Rate (%)")
plt.show()

### Interpretation: Support & Add-ons

Customers without support-related services tend to show higher churn. This suggests service depth and customer support may influence retention, either by improving customer experience or by increasing product stickiness.

**Business action:** for high-risk customers without tech support or security add-ons, test retention bundles that include support benefits rather than only price discounts.

## 15. Monthly Charges & Price Sensitivity

In [ ]:
df["MonthlyChargeBand"] = pd.qcut(
    df["MonthlyCharges"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
)

monthly_charge_churn = churn_rate_by("MonthlyChargeBand")
monthly_charge_churn

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", palette=["#2563eb", "#dc2626"], legend=False)
plt.title("Monthly Charges by Churn Outcome")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.show()

### Interpretation: Monthly Charges

Churned customers tend to have higher monthly charges. This may reflect price sensitivity, perceived value gaps, or customers comparing premium services against competitors.

**Business action:** monitor high-charge customers for dissatisfaction and offer value-based retention packages, not only discounts.

## 16. Combined High-Risk Segment

A practical retention strategy should combine risk factors instead of looking at one variable at a time.

In [ ]:
high_risk_segment = df[
    (df["Contract"] == "Month-to-month")
    & (df["tenure"] <= 12)
    & (df["PaymentMethod"] == "Electronic check")
]

pd.Series(
    {
        "customers_in_segment": len(high_risk_segment),
        "segment_churn_rate_%": round(high_risk_segment["ChurnFlag"].mean() * 100, 2),
        "avg_monthly_charge": round(high_risk_segment["MonthlyCharges"].mean(), 2),
    }
)

### Interpretation: Combined Risk Segment

Customers who are new, month-to-month, and using electronic check represent a concentrated risk segment. This is closer to how a business would actually use the analysis: define a priority group, then design an intervention.

**Business action:** build a retention campaign for this segment with onboarding support, billing migration incentives, and contract upgrade offers.

## 17. Business Intelligence Summary

| Finding | Why it matters | Recommended action |
|---|---|---|
| Month-to-month customers churn more | Low switching barrier | Offer annual contract incentives and loyalty bundles. |
| New customers churn more | Weak early relationship | Add onboarding, check-ins, and first-12-month retention journeys. |
| Electronic check users churn more | Possible billing friction or lower commitment | Encourage automatic payment migration with small credits. |
| High monthly charges relate to churn | Possible price sensitivity or value gap | Use value-based offers for premium customers. |
| Customers without support services churn more | Support depth may improve retention | Bundle tech support/security add-ons for high-risk users. |

## 18. Modeling Implications

The EDA supports the feature set used in the modeling pipeline:

- Keep contract, tenure, payment method, monthly charges, total charges, and service features.
- Drop `customerID` because it is only an identifier.
- Use `Churn` only as the target, never as an input feature.
- Evaluate models beyond accuracy because the target is imbalanced.

The next notebook should be `02_feature_engineering_and_modeling.ipynb`, covering preprocessing, train/test split, encoding, model comparison, confusion matrix, ROC-AUC, and business interpretation of model performance.